<div style="font-family:ui-monospace,'SF Mono','JetBrains Mono',Menlo,monospace;font-size:11px;letter-spacing:.24em;text-transform:uppercase;color:#A20025;margin:0 0 12px;">NeurIPS 2026 Education Track&nbsp;&middot;&nbsp;Generate, then verify</div>

<h1 style="font-family:Georgia,'Iowan Old Style','Palatino Linotype',serif;font-weight:500;font-size:33px;line-height:1.12;margin:0 0 16px;color:#1A1416;letter-spacing:-.01em;">Exercises 1 to 3. Sampling budget, and the control you forgot to run</h1>

<hr style="border:0;border-top:1px solid #A20025;opacity:.55;margin:0 0 4px;">

<em style="color:#6B6461;font-size:14.5px;">Module 1 (Refinement needs a referee) and module 2 (Straight paths, and testing your own explanation). Three tasks, 35 minutes.</em>

A sampler has a dial on it labelled "how many steps", and every paper that reports a
generative model has quietly chosen a setting for it. These three tasks are about
what that dial does to a number you can defend, and about the control you
have to run before you are allowed to explain the difference between two samplers.

We work on a two-dimensional toy: two half-moons standing in for a data
distribution, a small multilayer perceptron standing in for a denoiser, and the
energy distance standing in for every quality metric you will meet later. The toy is
deliberate. In two dimensions you can see the whole distribution at once and check
your metric against your eyes, which is exactly what you cannot do for a molecule in
three dimensions with nineteen heavy atoms. Learn the habit here, where it is
falsifiable, and carry it to module 3 where it is not.

You will do three things. First measure where a sampler breaks as you cut its
budget, and notice that it does not degrade gently. Then find the second thing that
differs between two samplers you were told differ in one way, hold it fixed, and
watch the conclusion change. Finally read a single ratio off a table and work out
what has to be stated alongside it for the ratio to mean anything.

<strong style="font-family:ui-monospace,'SF Mono','JetBrains Mono',Menlo,monospace;font-size:11px;letter-spacing:.16em;text-transform:uppercase;color:#A20025;">Objectives</strong>
<ol style="margin:8px 0 0;padding-left:22px;font-size:15.5px;color:#241c1d;">
<li style="margin:6px 0;">Measure sample quality against a budget and locate the budget where it fails</li>
<li style="margin:6px 0;">Name the confound in a two-sampler comparison and design the control that isolates it</li>
<li style="margin:6px 0;">State the threshold that a reported ratio between two samplers is relative to</li>
</ol>

<div style="margin:0 0 16px;padding:2px 0 2px 18px;border-left:2px solid #6B6461;">
<strong style="font-family:ui-monospace,'SF Mono','JetBrains Mono',Menlo,monospace;font-size:10.5px;letter-spacing:.16em;text-transform:uppercase;color:#6B6461;">Runtime</strong><br>
<span style="color:#3d3436;font-size:14.5px;">Everything in this notebook runs on a laptop CPU in well under a minute of compute. There is no download, no GPU, no API key and no network call; every number you produce comes from a file shipped inside this bundle.</span>
</div>

<div style="margin:0 0 16px;padding:2px 0 2px 18px;border-left:2px solid #A20025;">
<strong style="font-family:ui-monospace,'SF Mono','JetBrains Mono',Menlo,monospace;font-size:10.5px;letter-spacing:.16em;text-transform:uppercase;color:#A20025;">Note</strong><br>
<span style="color:#3d3436;font-size:14.5px;">Each task ends with a self-check cell. It tells you whether your answer sits inside the tolerance for that task and, when it does not, which way to look. It never prints the answer. The worked solutions live in <code>solutions/</code>; open them after you have an answer of your own, not before.</span>
</div>

In [ ]:
# Path anchor and toolkit import. This is the only setup cell in the notebook.
#
# The bundle looks like this:
#     NEURIPS2026_EDU_GENERATE_VERIFY/
#         gv_toolkit.py          <- the shared toolkit
#         data/                 <- every cached file the exercises read
#         exercises/            <- you are here
#         solutions/
# so the root is one level up from this notebook, and data/ hangs off the root.
# Resolving it positionally means the same cell works locally, on Colab, and in a
# grader's batch run, with no absolute path anywhere.
import sys
from pathlib import Path

HERE = Path.cwd()
ROOT = HERE.parent if HERE.name in {"exercises", "solutions"} else HERE
sys.path.insert(0, str(ROOT))

import gv_toolkit as gv

DATA = gv.DATA
print("bundle root:", ROOT.name, "| cached inputs:", len(gv.REQUIRED_FILES) + len(gv.SUPPORTING_FILES))

In [ ]:

# The pinned toy networks and the shared target. Both checkpoints ship in data/, so
# nothing is trained here and nothing can drift: the numbers you measure are the
# numbers your lecturer measured.
import numpy as np
import torch

torch.set_num_threads(4)

ddpm_net = gv.load_pinned("toy_ddpm_trained.pt")    # predicts noise
fm_net = gv.load_pinned("toy_fm_trained.pt")        # predicts velocity

BAND = 0.01          # the quality bar: energy distance to a fresh target draw
N_EVAL = 2000        # points per measurement
SEED = 0             # every sampling call in this notebook is seeded

print("networks loaded |",
      sum(p.numel() for p in ddpm_net.parameters()), "parameters each")
print("quality band:", BAND, "| samples per measurement:", N_EVAL)

<hr style="border:0;border-top:1px solid #A20025;opacity:.55;margin:34px 0 12px;">
<div style="font-family:ui-monospace,'SF Mono','JetBrains Mono',Menlo,monospace;font-size:11px;letter-spacing:.2em;text-transform:uppercase;color:#A20025;margin:0 0 8px;">X1&nbsp;&middot;&nbsp;introductory&nbsp;&middot;&nbsp;12 min&nbsp;&middot;&nbsp;M1&nbsp;&middot;&nbsp;O1, O2</div>
<h2 style="font-family:Georgia,'Iowan Old Style','Palatino Linotype',serif;font-weight:500;font-size:23px;margin:0 0 10px;color:#1A1416;">Find the step budget where the sampler breaks</h2>

The reverse process of a diffusion model is a loop, and the number of times you go
round it is yours to choose. Fewer iterations means a cheaper sample. The question
this task asks is what the discount costs, and the answer is not the one most people
predict.

**Before you run anything, write your prediction down.** In the cell below, set
`predicted_200` and `predicted_8` to the energy distance you expect at a 200-step
budget and at an 8-step budget. You will not be graded on being right. You will be
graded on having committed, because the gap between what you expected and what the
measurement says is the only part of this exercise that is hard to get from reading.

Two of the five energy distances are self-checked, at 200 and 8 steps, with a window
wide enough to absorb the seed-to-seed spread. The crossing budget is graded exactly,
because it is the same under every seed.

Then do three things.

1. Sample `N_EVAL` points from the ancestral DDPM sampler at each budget in
   `[200, 32, 16, 8, 4]` and measure the energy distance from each sample to a fresh
   draw from the target.
2. Report the smallest budget whose energy distance is still under the `BAND` of 0.01.
3. Write one sentence saying what you would have concluded from the scatter plots
   alone.

The plotting cell is written for you, so you can compare your eyes against your
metric.

In [ ]:
# X1, part 0. Commit to a prediction before you measure. Any number is a valid
# entry here; refusing to enter one is the only wrong answer.
predicted_200 = None      # your guess at the energy distance with 200 steps
predicted_8 = None        # your guess at the energy distance with 8 steps

if predicted_200 is None or predicted_8 is None:
    print("No prediction recorded. Fill both in before running the measurement: the "
          "surprise is the lesson, and you cannot be surprised by a number you never "
          "predicted.")
else:
    print(f"predicted: 200 steps -> {predicted_200}, 8 steps -> {predicted_8}")

In [ ]:
# X1, part 1. Measure. Fill in the loop body.
#
# gv.ddpm_sample(net, n, steps=K, seed=s) returns an (n, 2) tensor.
# gv.sample_target(n, seed=s) returns an (n, 2) numpy array from the target.
# gv.energy_distance(X, Y) takes two point clouds and returns a float.
#
# Use the same reference draw for every budget, so the only thing changing across
# the row is the budget itself.
BUDGETS = [200, 32, 16, 8, 4]
reference = gv.sample_target(N_EVAL, seed=1000)

measured = {}
for K in BUDGETS:
    # YOUR CODE HERE: draw N_EVAL samples at budget K with seed=SEED, measure the
    # energy distance to `reference`, and store it as measured[K].
    pass

for K in BUDGETS:
    print(f"  {K:>3} steps   energy distance {measured.get(K)}")

In [ ]:
# X1, part 2. Report the crossing budget: the smallest budget in BUDGETS whose
# energy distance is still under BAND.
crossing_budget = None      # YOUR CODE HERE

# X1, part 3. One sentence. What would the scatter plots alone have told you?
scatter_conclusion = ""     # YOUR SENTENCE HERE

In [ ]:
# The plots. Your eyes against your metric, at the same five budgets.
import matplotlib.pyplot as plt

target_cloud = gv.sample_target(2000, seed=99)
fig, axes = plt.subplots(1, len(BUDGETS), figsize=(14, 3))
for ax, K in zip(axes, BUDGETS):
    s = gv.ddpm_sample(ddpm_net, 1500, steps=K, seed=1).numpy()
    ax.scatter(target_cloud[:, 0], target_cloud[:, 1], s=4, alpha=0.18, color="#9aa7b3")
    ax.scatter(s[:, 0], s[:, 1], s=4, alpha=0.5, color="#A20025")
    ax.set_title(f"{K} steps", fontsize=11)
    ax.set_xlim(-3, 3); ax.set_ylim(-3, 3)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_aspect("equal")
fig.suptitle("Ancestral DDPM at five reverse-step budgets", y=1.04, color="#A20025")
fig.tight_layout()
plt.show()

<div style="font-size:12.5px;color:#6B6461;margin:6px 0 14px;">Ancestral DDPM samples in crimson against the target in grey, at budgets of 200, 32, 16, 8 and 4 reverse steps. The grey cloud is the same in every panel. Read the panels left to right and decide, by eye alone, at which panel you would have stopped trusting the sampler. Then compare that panel against the budget your energy-distance measurement picked out. The two answers are not usually the same, and the gap between them is what a quantitative check buys you.</div>

In [ ]:
# X1 self-check. Tells you whether each answer is inside tolerance. Never the answer.
gv.check_value("X1.crossing", crossing_budget)
gv.check_value("X1.ed200", measured.get(200))
gv.check_value("X1.ed8", measured.get(8))
gv.check_written("X1.scatter", scatter_conclusion,
                  "What would the five panels alone have told you about where to "
                  "stop trusting the sampler?")
gv.checklist_summary("X1")

<hr style="border:0;border-top:1px solid #A20025;opacity:.55;margin:34px 0 12px;">
<div style="font-family:ui-monospace,'SF Mono','JetBrains Mono',Menlo,monospace;font-size:11px;letter-spacing:.2em;text-transform:uppercase;color:#A20025;margin:0 0 8px;">X2&nbsp;&middot;&nbsp;intermediate&nbsp;&middot;&nbsp;15 min&nbsp;&middot;&nbsp;M2&nbsp;&middot;&nbsp;O3, O6</div>
<h2 style="font-family:Georgia,'Iowan Old Style','Palatino Linotype',serif;font-weight:500;font-size:23px;margin:0 0 10px;color:#1A1416;">Two ways to be fooled by a validator</h2>

Every quality number has a floor it cannot go below, and every comparison between
two methods has a variable somebody forgot to hold fixed. This task is one of each.

**Part one, the floor.** Draw two independent samples of `N_EVAL` points from the
target itself and measure the energy distance between them. Neither sample came from
a model. Report the value, and say what it implies about the 0.01 band we have been
treating as a quality bar.

**Part two, the control.** You have been told that flow matching beats ancestral
diffusion at low step counts because its paths through space are straighter. That
explanation may even be correct. But there is a second thing that differs between
those two samplers, and until you hold it fixed you cannot attribute the gap to path
straightness at all.

Name that second thing. Then use the shipped eta sweep to hold it fixed, and report
whether the gap between the two samplers narrows or widens once you do.

The eta sweep is one budget, one network, one dial. The dial is the amount of noise
the DDIM sampler injects at each reverse step: eta of 0 is a fully deterministic
solve and eta of 1 reproduces the ancestral update exactly. Sweeping it moves you
from one sampler family to the other without changing anything else.

<div style="margin:0 0 16px;padding:2px 0 2px 18px;border-left:2px solid #A20025;">
<strong style="font-family:ui-monospace,'SF Mono','JetBrains Mono',Menlo,monospace;font-size:10.5px;letter-spacing:.16em;text-transform:uppercase;color:#A20025;">Note</strong><br>
<span style="color:#3d3436;font-size:14.5px;">Part two has a hint, and using it costs you nothing. The hint is one question: besides the shape of the path, what else is different about how these two samplers take a step?</span>
</div>

In [ ]:
# X2, part 1. The two-sample floor.
#
# Both clouds come from gv.sample_target. Use different seeds. Nothing here is a
# model output, so whatever you measure is the metric's own noise at this sample size.
floor_values = []
for i in range(5):
    # YOUR CODE HERE: draw two independent target samples of N_EVAL points with
    # different seeds, measure the energy distance between them, append it.
    pass

two_sample_floor = float(np.mean(floor_values)) if floor_values else None
print("per-pair values:", [round(v, 6) for v in floor_values])
print("mean two-sample floor:", two_sample_floor)

# One sentence: what does this floor say about the 0.01 band?
floor_interpretation = ""    # YOUR SENTENCE HERE

In [ ]:
# X2, part 2a. Name the confound in one word or short phrase, before you look at
# the sweep. What differs between the two samplers besides the shape of the path?
confound = ""     # YOUR ANSWER HERE

In [ ]:
# X2, part 2b. Hold it fixed and measure.
#
# data/toy_eta_sweep.csv holds the energy distance at a fixed 8-step budget for
# six settings of eta, over five seeds each. eta = 1.0 is the ancestral update;
# eta = 0.0 is the deterministic one. Both use the SAME network as each other.
eta_sweep = gv.read_metrics_csv(DATA / "toy_eta_sweep.csv")
eta_mean = eta_sweep.groupby("eta").energy_distance.mean()
print(eta_mean.round(5).to_string())

# YOUR CODE HERE: what does removing the sampling noise entirely cost, as a factor?
# (the eta = 0.0 mean divided by the eta = 1.0 mean)
noise_removal_factor = None

# And now the comparison the stem asks for. The shipped sampler sweep has flow
# matching alongside both diffusion samplers at every budget.
sweep = gv.read_metrics_csv(DATA / "toy_sampler_sweep.csv")
mean_energy = sweep.groupby(["sampler", "steps"]).energy_distance.mean().unstack()
print()
print(mean_energy.round(5).to_string())

# Does the flow-matching advantage narrow or widen once the sampler family is held
# fixed at the deterministic end? Answer with the single word "narrows" or "widens".
gap_direction = ""     # YOUR ANSWER HERE

In [ ]:
# X2 self-check.
gv.check_value("X2.floor", two_sample_floor)
gv.check_text("X2.confound", confound)
gv.check_value("X2.factor", noise_removal_factor)
gv.check_text("X2.direction", gap_direction)
gv.check_written("X2.floor_meaning", floor_interpretation,
                  "How far above the metric's own two-sample floor does the 0.01 "
                  "band sit, and does that make it a tight bar or a generous one?")
gv.checklist_summary("X2")

<hr style="border:0;border-top:1px solid #A20025;opacity:.55;margin:34px 0 12px;">
<div style="font-family:ui-monospace,'SF Mono','JetBrains Mono',Menlo,monospace;font-size:11px;letter-spacing:.2em;text-transform:uppercase;color:#A20025;margin:0 0 8px;">X3&nbsp;&middot;&nbsp;introductory&nbsp;&middot;&nbsp;8 min&nbsp;&middot;&nbsp;M2&nbsp;&middot;&nbsp;O4, O5</div>
<h2 style="font-family:Georgia,'Iowan Old Style','Palatino Linotype',serif;font-weight:500;font-size:23px;margin:0 0 10px;color:#1A1416;">One network, two jobs</h2>

The diffusion and flow-matching toy networks use the
same architecture, the same width, the same depth, and the same number of parameters.
`FieldMLP` maps a point and a time to a two-dimensional vector, and that is the whole
of it. What differs is what the vector is asked to be.

Three parts.

1. Write the one line of each training loop that differs. Both loops are printed
   below with that line blanked out.
2. Read the crossing budget for each sampler off the shipped sweep at the 0.01 band,
   and state the ratio.
3. Say why quoting that ratio without naming the band would be meaningless.

Part three is the graded part, and it is the first appearance of a move that recurs
in X6 with fingerprints and in X8 with retention: a comparison is only defined
relative to the threshold you compared at.

In [ ]:
# X3, part 1. The two training loops, side by side, with one line blanked in each.
#
# --- diffusion (the Notebook 1 loop) ---------------------------------------
#     x0  = target_batch
#     t   = random timestep
#     eps = torch.randn_like(x0)
#     xt  = q_sample(x0, t, eps)             # noise x0 up to level t
#     pred = net(xt, t / T)
#     loss = ???                             # <-- LINE A
#
# --- flow matching (the Notebook 2 loop) -----------------------------------
#     x1 = target_batch                      # data end
#     x0 = torch.randn_like(x1)              # noise end
#     t, xt, ut = FM.sample_location_and_conditional_flow(x0, x1)
#     vt = net(xt, t)
#     loss = ???                             # <-- LINE B
#
# Write each line as a string. What quantity is the network's output regressed onto?
line_A_diffusion = ""      # YOUR ANSWER HERE
line_B_flow = ""           # YOUR ANSWER HERE

In [ ]:
# X3, part 2. Read the crossing budgets off the sweep.
#
# `mean_energy` is already in memory from X2: seed-mean energy distance indexed by
# sampler and budget. The crossing budget is the smallest budget whose seed-mean is
# under BAND and stays under it at every larger budget. Exclude the 1-step column:
# for the respaced diffusion samplers a single step is a degenerate no-op, so its
# score describes the input noise rather than the sampler.


def crossing_budget_of(sampler, band=BAND):
    """Return the smallest budget above 1 that is under `band` and stays under it."""
    row = mean_energy.loc[sampler].dropna()
    row = row[row.index > 1]
    # YOUR CODE HERE: return the smallest such budget, or None if there is none.
    return None


cross_fm = crossing_budget_of("fm_euler")
cross_ddpm = crossing_budget_of("ddpm_ancestral")
budget_ratio = None        # YOUR CODE HERE: ddpm crossing divided by fm crossing

print(f"flow matching   crosses at {cross_fm}")
print(f"ancestral DDPM  crosses at {cross_ddpm}")
print(f"ratio           {budget_ratio}")

# For part 3, look at what the same two samplers do at the largest budget.
print()
print("at 200 steps:  fm_euler", round(float(mean_energy.loc["fm_euler", 200]), 5),
      "| ddpm_ancestral", round(float(mean_energy.loc["ddpm_ancestral", 200]), 5))

In [ ]:
# X3, part 3. Why is the ratio meaningless without naming the band?
band_dependence = ""       # YOUR SENTENCES HERE (two at most)

In [ ]:
# X3 self-check.
gv.check_value("X3.cross_fm", cross_fm)
gv.check_value("X3.cross_ddpm", cross_ddpm)
gv.check_value("X3.ratio", budget_ratio)
gv.check_written("X3.line_A", line_A_diffusion, "What is the diffusion loss regressed onto?")
gv.check_written("X3.line_B", line_B_flow, "What is the flow-matching loss regressed onto?")
gv.check_written("X3.band", band_dependence,
                  "The two curves are not parallel. What happens to the ratio at a "
                  "different band?")
gv.checklist_summary()

<hr style="border:0;border-top:1px solid #A20025;opacity:.55;margin:34px 0 12px;">
<h2 style="font-family:Georgia,'Iowan Old Style','Palatino Linotype',serif;font-weight:500;font-size:23px;margin:0 0 10px;color:#1A1416;">Before you move on</h2>

Three answers from this notebook get reused. The crossing budgets are the numbers
module 3 compares against when the sampler starts producing molecules instead of
points. The two-sample floor is the reason module 4 refuses to report a validity
percentage without its batch size. And the confound you named in X2 is the same
shape of mistake as the one waiting in X5, where a verification result will look
sound for a reason that has nothing to do with the molecules.

Next: `EX2_molecules_and_their_checks.ipynb`.